<a href="https://colab.research.google.com/github/PiKr-Salmon/CNN-Shape-Texture-/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# ตั้งค่า Device (GPU/CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
class CannyEdgeTransform:
    """แปลงภาพเป็น Canny Edges เพื่อลบสีและ Local Texture ออก เหลือเพียง Global Shape/Structure"""
    def __init__(self, low_threshold=100, high_threshold=200):
        self.low_threshold = low_threshold
        self.high_threshold = high_threshold

    def __call__(self, img):
        img_np = np.array(img)
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, self.low_threshold, self.high_threshold)
        edges_3ch = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB) # ขยายเป็น 3 ช่องสีเพื่อให้เข้าโมเดลได้
        return Image.fromarray(edges_3ch)

# 1. Pipeline ภาพปกติ (Normal RGB)
transform_normal = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Pipeline ภาพดัดแปลง (Canny Edge / Shape-only)
transform_edge = transforms.Compose([
    transforms.Resize((224, 224)),
    CannyEdgeTransform(low_threshold=80, high_threshold=160),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transform สำหรับแสดงผลรูปภาพ (Un-normalize)
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

In [ ]:
DATA_DIR = './data'

# โหลด Test Set แบบภาพปกติ
testset_normal = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_normal)
testloader_normal = DataLoader(testset_normal, batch_size=32, shuffle=False, num_workers=2)

# โหลด Test Set แบบภาพขอบ (Edge / Shape Test Set)
testset_edge = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_edge)
testloader_edge = DataLoader(testset_edge, batch_size=32, shuffle=False, num_workers=2)

# โหลด Train Set แบบภาพขอบ (สำหรับนำไป Fine-tune โมเดลใน Condition 2)
trainset_edge = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform_edge)
trainloader_edge = DataLoader(trainset_edge, batch_size=32, shuffle=True, num_workers=2)

classes = testset_normal.classes
print(f"Classes: {classes}")

In [ ]:
def evaluate_model(model, dataloader, device):
    """ฟังก์ชันวัดค่า Accuracy บน DataLoader ที่กำหนด"""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100.0 * correct / total

def create_resnet18():
    """สร้าง ResNet-18 Pre-trained และเปลี่ยน FC layer ให้ตรงกับ 10 คลาส"""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(device)

In [ ]:
print("--- 1. Evaluating Baseline Pre-trained ResNet-18 ---")
baseline_model = create_resnet18()

# วัดบนภาพปกติ
acc_baseline_normal = evaluate_model(baseline_model, testloader_normal, device)
print(f"Baseline Accuracy on Normal Images: {acc_baseline_normal:.2f}%")

# วัดบนภาพ Canny Edge (ลบ Texture ออก)
acc_baseline_edge = evaluate_model(baseline_model, testloader_edge, device)
print(f"Baseline Accuracy on Edge Images (Texture Removed): {acc_baseline_edge:.2f}%")

In [ ]:
print("\n--- 2. Fine-tuning ResNet-18 on Edge Dataset ---")
finetuned_model = create_resnet18()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(finetuned_model.parameters(), lr=0.0001)

# Fine-tune เป็นเวลา 3 Epochs (สามารถเพิ่มเป็น 5-10 Epochs ได้ถ้าต้องการ Accuracy สูงขึ้น)
epochs = 3
finetuned_model.train()

for epoch in range(epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(trainloader_edge):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = finetuned_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i + 1) % 300 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(trainloader_edge)}], Loss: {running_loss/300:.4f}")
            running_loss = 0.0

# ประเมินผลหลัง Fine-tune
acc_finetuned_normal = evaluate_model(finetuned_model, testloader_normal, device)
acc_finetuned_edge = evaluate_model(finetuned_model, testloader_edge, device)

print(f"\nFine-tuned Accuracy on Normal Images: {acc_finetuned_normal:.2f}%")
print(f"Fine-tuned Accuracy on Edge Images: {acc_finetuned_edge:.2f}%")

In [ ]:
# แสดงตารางเปรียบเทียบ
print("\n================== EXPERIMENTAL RESULTS ==================")
print(f"{'Model Condition':<25} | {'Normal Accuracy (%)':<20} | {'Edge Accuracy (%)':<20}")
print("-" * 70)
print(f"{'Baseline (Pre-trained)':<25} | {acc_baseline_normal:<20.2f} | {acc_baseline_edge:<20.2f}")
print(f"{'Fine-tuned (Edge)':<25} | {acc_finetuned_normal:<20.2f} | {acc_finetuned_edge:<20.2f}")
print("==========================================================")

# พล็อตกราฟเปรียบเทียบ
categories = ['Normal Test Set', 'Edge Test Set (Shape Only)']
baseline_scores = [acc_baseline_normal, acc_baseline_edge]
finetuned_scores = [acc_finetuned_normal, acc_finetuned_edge]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline (ImageNet Pre-trained)', color='#4C72B0')
rects2 = ax.bar(x + width/2, finetuned_scores, width, label='Fine-tuned (Edge Trained)', color='#DD8452')

ax.set_ylabel('Accuracy (%)')
ax.set_title('CNN Performance: Normal vs. Shape-Only (Edge) Test Sets')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

# บันทึกรูปภาพไว้สำหรับใส่ในรายงาน
plt.savefig('experimental_results.png', dpi=300, bbox_inches='tight')
plt.show()